![Pythonista](img/pythonista.png)

# *Terraform* para *Compute Engine* y red.

Esta sesión traslada el enfoque de despliegue seguro a infraestructura más cercana a *Linux* tradicional: VM, red, reglas de firewall y base para endurecimiento posterior.

## Objetivos.

- Entender el diseño de VM, VPC y reglas base de red.
- Relacionar esta ruta con despliegue *Linux* en entorno de test.
- Distinguir entre infraestructura base ya implementada y extensiones de infraestructura.
- Aplicar una secuencia operativa mínima: `plan` → `apply` → verificación → rollback.

En el codebase actual, la base de *Terraform* prioriza *Artifact Registry* y *Cloud SQL*. La capa de [*Compute Engine*](https://cloud.google.com/compute) y red se trabaja como extensión didáctica para escenarios de operación *Linux*.


## Recordatorio para principiantes de *IaC*.

Cuando pases de serverless a VM y red, *Terraform* sigue la misma lógica:

- declaras topología (VPC, subredes, reglas, instancia);
- revisas impacto con `plan`;
- aplicas cambios de forma controlada con `apply`.

Esto evita "configuración artesanal" en consola y permite auditar cada cambio de infraestructura como parte del flujo de *CI/CD*.


## Base para despliegue en *Linux*.

Cuando el destino es *Compute Engine*, el *pipeline* debe coordinarse con decisiones de red y sistema operativo: reglas de firewall, apertura de puertos, cuentas, acceso SSH y rutas de despliegue.

En este curso se enfatiza una idea: un despliegue seguro en *Linux* no empieza al conectarse por SSH; empieza al modelar correctamente la infraestructura que recibirá la aplicación.

### Secuencia operativa mínima recomendada.

1. `terraform plan` y revisión humana del impacto (red, puertos, identidad).
2. `terraform apply` en entorno controlado.
3. Verificación técnica: conectividad por IAP, estado de servicio y *health checks*.
4. Rollback de infraestructura si el cambio rompe conectividad o endurecimiento esperado.

### Criterios de hardening de base.

- Evitar IP pública en la VM cuando no sea estrictamente necesaria.
- Limitar SSH a IAP y usar *OS Login* en lugar de llaves manuales.
- Separar identidad de despliegue e identidad de ejecución.
- Aplicar mínimo privilegio en IAM y revisar reglas de firewall por etiqueta de instancia.


In [ ]:
hcl = """
resource "google_compute_network" "vpc" {
  name                    = "${var.app_name}-vpc"
  auto_create_subnetworks = false
}

resource "google_compute_subnetwork" "subnet" {
  name          = "${var.app_name}-subnet"
  ip_cidr_range = "10.0.1.0/24"
  region        = var.region
  network       = google_compute_network.vpc.id
}

resource "google_compute_firewall" "ssh_iap" {
  name    = "${var.app_name}-allow-ssh-iap"
  network = google_compute_network.vpc.name

  allow {
    protocol = "tcp"
    ports    = ["22"]
  }

  # Solo IAP puede abrir túneles SSH hacia instancias etiquetadas.
  source_ranges = ["35.235.240.0/20"]
  target_tags   = ["ssh-iap"]
}

resource "google_service_account" "runtime" {
  account_id   = "${var.app_name}-runtime"
  display_name = "Runtime identity for Linux deployment"
}

resource "google_compute_instance" "app" {
  name         = "${var.app_name}-vm"
  machine_type = "e2-micro"
  zone         = "${var.region}-a"
  tags         = ["ssh-iap"]

  boot_disk {
    initialize_params { image = "debian-cloud/debian-12" }
  }

  network_interface {
    subnetwork = google_compute_subnetwork.subnet.id
    # Sin access_config => sin IP pública.
  }

  metadata = {
    enable-oslogin = "TRUE"
  }

  shielded_instance_config {
    enable_secure_boot = true
    enable_vtpm        = true
    enable_integrity_monitoring = true
  }

  # Identidad de ejecución separada del actor de despliegue.
  service_account {
    email = google_service_account.runtime.email
    scopes = [
      "https://www.googleapis.com/auth/logging.write",
      "https://www.googleapis.com/auth/monitoring.write",
    ]
  }
}
"""

print(hcl)


<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>